This notebook attempts to make an ML model using Random forest and LightGBM to predict the vertical drop of plasma curret using several features. It gives the probability with a 50ms future window. It uses data from the FAIR MAST public dataset, and uses the time series data from the range of shots 18500-30500.

In [ ]:
import zarr
import xarray as xr
import numpy as np
import pandas as pd
import fastparquet

import matplotlib.pyplot as plt
import os
import warnings
import random

from sklearn.ensemble import RandomForestClassifier 
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score, RocCurveDisplay
from lightgbm import LGBMClassifier

from concurrent.futures import ThreadPoolExecutor, as_completed
import threading

warnings.filterwarnings('ignore', category=RuntimeWarning)
warnings.filterwarnings('ignore', module='zarr')


In [ ]:

def get_store(shot_id: int):
    """Retrieves data for a given shot ID from the MAST S3 bucket
    and returns a Zarr store object.
    
    Parameters:
        shot_id : int
            The ID of the shot to retrieve data for.
    
    Returns:
        store : zarr.storage.FsspecStore
            A Zarr store object containing the data for the specified shot.
    """
    try:
        endpoint_url = 'https://s3.echo.stfc.ac.uk'
        url = f's3://mast/level2/shots/{shot_id}.zarr'

        # Get a handle to the remote file
        store = zarr.storage.FsspecStore.from_url(
            url,
            storage_options=dict(
                protocol='simplecache',
                target_protocol="s3",
                cache_storage='.cache', # stores the cached data locally
                target_options=dict(
                    anon=True, # anonymous connection to S3
                    endpoint_url=endpoint_url, 
                    asynchronous=True) # doesn't stop notebook
            )
        )

        return store
    
    except Exception as e:
        print(f"Failed to load store: {e}")
        return None


In [ ]:
store = get_store(30400)

# print the tree structure of the data
root = zarr.open(store, mode="r")
print(root.tree())


In [ ]:
# thread-safe lock for writing to zarr
zarr_lock = threading.Lock()

def process_shot(shot_id: int) -> xr.Dataset | None:
    """Processes a single shot and returns the dataset or None.
    
    Parameters:
        shot_id : int
    
    Returns:
        xr.Dataset | None
            Dataset containing the processed shot parameters, or None if processing fails.
    """
    try:
        store = get_store(shot_id)
        if store is None:
            return None

        # open all groups once per shot
        groups = {}
        
        for group in ['summary', 'equilibrium']:
            try:
                groups[group] = xr.open_zarr(store, group=group)
            except:
                   groups[group] = None

        if groups['summary'] is None:
            return None

        common_time = groups['summary']['time']

        parameters = {
            'ip':              ('summary', 'ip'),
            'power_nbi':       ('summary', 'power_nbi'),
            'power_radiated':  ('summary', 'power_radiated'),
            'greenwald':       ('summary', 'greenwald_density'),

            'wmhd':            ('equilibrium', 'wmhd'),
            'vloop':           ('equilibrium', 'vloop_static'),
            'q95':             ('equilibrium', 'q95'),
            'beta_tor_normal': ('equilibrium', 'beta_tor_normal'),
            'li':              ('equilibrium', 'li'),
            'elong':           ('equilibrium', 'elongation'),
        }

        optional_vars = {'greenwald', 'elong'}
        profile_ds = xr.Dataset()

        for key, (group, val) in parameters.items():
            try:
                if groups[group] is None:
                    if key not in optional_vars:
                        return None
                    continue

                variable_data = groups[group][val]

                if 'time' not in variable_data.dims:
                    if key not in optional_vars:
                        return None
                    continue

                # interpolate to common time axis
                variable_data = variable_data.interp(
                    time=common_time, method='linear')

                if variable_data.isnull().all():
                    if key not in optional_vars:
                        return None
                    continue

                profile_ds[key] = variable_data

            except Exception:
                if key not in optional_vars:
                    return None
                continue

        # check required variables present
        required = ['ip', 'power_nbi', 'wmhd', 'power_radiated', 
                    'vloop', 'q95', 'beta_tor_normal', 'li']
        
        required_present = all(
            k in profile_ds.data_vars and 
            not profile_ds[k].isnull().all().item()
            for k in required)

        if not required_present:
            return None

        profile_ds = profile_ds.assign_coords(shot_id=shot_id)
        return profile_ds

    except Exception as e:
        print(f"Shot {shot_id} failed: {e}")
        return None


In [ ]:
# thread-safe lock for writing to zarr
zarr_lock = threading.Lock()

def get_parameters_parallel(shot_min: int, shot_max: int, max_workers: int = 4) -> None:
    """Processes shots in parallel using threads. Prevents race conditions
    by locking zarr writes while multiple threads process different shots.
    
    Parameters:
        shot_min : int
            Minimum shot ID to process.
        shot_max : int
            Maximum shot ID to process.
        max_workers : int
            Number of parallel threads. Default is 4.
        
    Returns:
        None
    """
    os.makedirs('mast_profiles', exist_ok=True)
    zarr_path = f'mast_profiles/mast_parameters_{shot_min}_{shot_max}.zarr'

    shot_ids = range(shot_min, shot_max + 1)
    succeeded = 0
    failed = 0

    # submit all shots to thread pool
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        
        futures = {
            executor.submit(process_shot, shot_id, zarr_path): shot_id 
            for shot_id in shot_ids
        }

        for future in as_completed(futures):
            shot_id = futures[future]
            try:
                result = future.result()

                if result is not None:
                    # zarr isn't thread-safe for writing
                    # so we lock while writing
                    with zarr_lock:
                        result.to_zarr(
                            zarr_path, 
                            mode='a', 
                            group=f'shot_{shot_id}',
                            consolidated=False
                        )
                    succeeded += 1
                    print(f"Shot {shot_id} succeeded ({succeeded} total)")
                else:
                    failed += 1

            except Exception as e:
                print(f"Shot {shot_id} failed: {e}")
                failed += 1

    print(f"\nDone: {succeeded} succeeded, {failed} failed")


Using threading to collect time profiles from dataset more efficiently.

In [ ]:
def read_file(shot_min: int, shot_max: int, max_workers: int = 4) -> str:
    """Reads the zarr file for a shot range, or creates it if not found.
    If the file does not exist, calls get_parameters_parallel to generate it.
    
    Parameters:
        shot_min : int
            Minimum shot ID.
        shot_max : int
            Maximum shot ID.
        max_workers : int
            Number of parallel threads. Default is 4.
    
    Returns:
        zarr_path : str
            File path to the zarr file containing the parameters.
    """

    zarr_path  = f'mast_profiles/mast_parameters_{shot_min}_{shot_max}.zarr'

    # checks if file exists
    if os.path.exists(zarr_path):
        parameters = xr.open_zarr(zarr_path, consolidated=False)
        return zarr_path
    
    else:
        # makes file and then reads it
        get_parameters_parallel(shot_min, shot_max, max_workers=max_workers)
        try:
            parameters = xr.open_zarr(zarr_path, consolidated=False)
            print("File creation successful")
            return zarr_path
        except Exception as e:
            print(f"File was not made, likely due to missing data: {e}")
            return None


In [ ]:
shot_min = 18500
shot_max = 30500

zarr_path = read_file(shot_min, shot_max, max_workers=4)

root = zarr.open(zarr_path, mode="r")
print(root.tree())


In [ ]:
def plotting_profiles(shot_id: str, zarr_path: str) -> None:
    """Plots all variables for a shot over time in a 5x2 subplot grid.
    
    Parameters:
        shot_id : str
            The shot ID to plot (e.g., 'shot_24073').
        zarr_path : str
            File path to the zarr file containing the data.
    
    Return:
        None
    """
    
    variables = ['ip', 'wmhd', 'power_nbi', 'power_radiated', 
                'q95', 'beta_tor_normal', 'li', 'elong', 'vloop', 'greenwald']
    
    parameters = xr.open_zarr(zarr_path, group=shot_id, consolidated=False)

    fig, axes = plt.subplots(5, 2, figsize=(14, 16), sharex=True)
    axes = axes.flatten()

    labels = {
        'ip': 'Plasma Current (A)',
        'wmhd': 'WMHD - Stored Energy (J)',
        'power_nbi': 'NBI Power (W)',
        'power_radiated': 'Radiated Power (W)',
        'q95': 'Safety Factor q95',
        'beta_tor_normal': 'Normalised Beta',
        'li': 'Internal Inductance',
        'elong': 'Elongation',
        'vloop': 'Loop Voltage (V)',
        'greenwald': 'Greenwald Density (m⁻³)'
    }

    for i, var in enumerate(variables):
        data = parameters[var].squeeze().dropna(dim='time')
        axes[i].plot(data.time.clip(min=0), data)
        axes[i].set_ylabel(labels[var])
        axes[i].set_xlabel('Time (s)')

    fig.suptitle(f'{shot_id}: Plasma Parameters', fontsize=14)
    plt.tight_layout()

    plt.show()


In [ ]:
shot_min = 18500
shot_max = 30500
shot_id = f'shot_{24073}'

zarr_path = read_file(shot_min, shot_max)
plotting_profiles(shot_id, zarr_path)


Above shows a time series profile of the parameters we'll be using to train the ML model with. Each parameter has been interpolated to the summary group's time coordinate to ensure time resoultions are the same.

In [ ]:
def review__labels(shot_ids: list, zarr_path: str, sample_size: int = 5) -> tuple:
    """Plots a sample of shots for manual verification of labels.
    Press Enter to confirm label, or type 'n' to mark as incorrect.
    
    Parameters:
        shot_ids : list
            List of shot IDs to review.
        zarr_path : str
            File path to zarr file containing data.
        sample_size : int
            Number of shots to sample for review. Default is 5.
    
    Returns:
        confirmed : list
            Shot IDs confirmed as correct by the user.
        rejected : list
            Shot IDs marked as incorrect by the user.
    """
    confirmed = []
    rejected = []
    sample = []

    # chooses random sample from shot_ids
    for i in range(sample_size):
        sample.append(random.choice(shot_ids))

    for shot_id in sample:
        try:
            
            plotting_profiles(shot_id, zarr_path)

            response = input(f"{shot_id} correct labelling? [y]/n: ").strip().lower()
            
            if response == 'n':
                rejected.append(shot_id)
                print(f"{shot_id} marked as incorrect labelling")
            else:
                confirmed.append(shot_id)
                print(f"{shot_id} confirmed as correct labelling")

        except Exception as e:
            print(f"Could not load shot {shot_id}: {e}")

    print(f"\nCorrect labelling: {len(confirmed)}")
    print(f"Incorrect labelling: {len(rejected)}")
    
    return confirmed, rejected


In [ ]:
def is_bad_data(zarr_path: str) -> str | None:
    """Determines whether a shot is bad data, the conditions for bad
    data are: not nbi heated, large negative values in nbi, ip, and 
    power radiated, and no significant decrease in plasma current at the end
    of the shot.

    Parameters:
        zarr_path : str
            File path to the zarr file containing the data.
    
    Returns:
        label_path : str
            file path containing the shots for good data
    """

    root = zarr.open_group(store=zarr_path, mode='r')
    os.makedirs('mast_profiles_ML_data', exist_ok=True)

    # path for the good data shot ids to be stored 
    label_path = 'mast_profiles_ML_data/good_data.csv'
    good_shots = []
    n_success = 0

    if os.path.exists(label_path):
        print("Good data file already exists, skipping check")
        return label_path
    else:
        for shot in root:

            try:
                data = xr.open_zarr(zarr_path, group=shot, consolidated=False)
                ip = data['ip']
                power_nbi = data['power_nbi']
                power_radiated = data['power_radiated']
                
                if float(ip.mean()) < 0:
                    print(f'{shot}: negative current shot, skipping')
                    continue
                
                elif np.abs(float(power_radiated.min())) > np.abs(float(power_radiated.max())):
                    print(f'{shot}: unphysical shot, skipping')
                    continue
                
                # get time when ip starts levelling out
                mask = data["ip"] > 0.7 * data["ip"].median()        
                start_time = data["ip"].time.where(mask, drop=True).min()

                ip_late = data["ip"].sel(time=slice(start_time, None))
                nbi_late = power_nbi.where(power_nbi.time > start_time, drop=True)

                # checks first point where ip is off
                try:
                    ip_drop_time = float(ip_late.time.where(ip_late < 0.3 * ip_late.max(), drop=True)[0])
                except IndexError:
                    print(f'{shot}: no plasma drop, skipping shot')
                    continue # no plasma decrease

                # only use nbi heated shots
                is_nbi_heated = bool(nbi_late.max() > 1e5)
                if not is_nbi_heated:
                    print(f'{shot}: nbi heating not used, skipping shot')
                    continue
                
                else: 
                    good_shots.append(shot)
                    n_success += 1
                    print(f'{shot}: good data ({n_success} total)')
            
            except Exception as e:
                print(f'{shot} failed: {e}')
                continue
        pd.DataFrame(good_shots, columns=['shot_id']).to_csv(label_path, index=False)
        return label_path


Clearing non_NBI heated shots from the dataset, in order to minimise confusion for the model. Furthermore, cleaning shots containing dubious measurements with unphysical values in power NBI, power Radiated, and plasma current.

In [ ]:

def extracting_features(zarr_path: str, label_path: str, 
                         window_size: float = 0.02, future_window_size: float = 0.05) -> str:
    """Extracts time-windowed features from shots for machine learning.
    Creates sliding windows of specified size and step from flat-top to shot end.
    
    Parameters:
        data_path : str
            zarr path to the shot data
        label_path : str
            File path to the disruption labels CSV.
        window_size : float
            Size of each time window in seconds. Default is 20ms.
        future_window_size : float
            Time step between windows in seconds. Default is 50ms.
    
    Returns:
        file_path : str
            File path to data.
    """
    
    os.makedirs('mast_profiles_ML_data', exist_ok=True)
    shot_df = pd.read_csv(label_path)

    # file for the data to be stored
    file_path = 'mast_profiles_ML_data/features_extraction.parquet'
    n_success = 0

    if os.path.exists(file_path):
        print("Feature file already exists, skipping extraction")
        return file_path
    else:
        for shot in shot_df['shot_id']:
            try:
                data = xr.open_zarr(zarr_path, group=shot, consolidated=False)

                features = ['ip', 'wmhd', 'power_nbi', 'power_radiated', 
                    'q95', 'beta_tor_normal', 'li', 'elong', 'vloop', 'greenwald']

                # gets the first point where theshold is met
                mask = data["ip"] > 0.7 * data["ip"].median()        
                start_time = data["ip"].time.where(mask, drop=True).min()
                ip_trim = data["ip"].sel(time=slice(start_time, None))

                dt = float(data['ip'].time.diff('time').median())
                window_steps = max(int(window_size / dt), 2)
                future_steps = max(int(future_window_size / dt), 2)

                X = xr.Dataset()

                # gets data for features
                for var in features:
                    var_data = data[var].sel(time=slice(start_time, None))
                    X[f'{var}_mean'] = var_data.rolling(time=window_steps).mean()
                    X[f'{var}_std']  = var_data.rolling(time=window_steps).std()
                    X[f'{var}_grad'] = var_data.diff('time').rolling(time=window_steps).mean()
                    X[f'{var}_min_grad'] = var_data.diff("time").rolling(time=window_steps).min()
                    X[f'{var}_range'] = var_data.rolling(time=window_steps).max() - var_data.rolling(time=window_steps).min()

                # shifts past window, to future window, shifts forward 5ms
                future_min = ip_trim.rolling(time=future_steps).min().shift(time=-future_steps)
                past_max = ip_trim.rolling(time=window_steps).max()

                # if ip dros by 40% in next 5ms
                drop_fraction = (past_max - future_min) / past_max
                Y = drop_fraction > 0.4

                X['label'] = Y
                X['shot_id'] = shot
            
                df = X.to_dataframe().dropna()
                
                df = df.astype({col: 'float32' for col in df.columns if col not in ['label', 'shot_id']})
                df['label'] = df['label'].astype(bool)
                df['shot_id'] = df['shot_id'].astype(str)
                
                # append to parquet
                if os.path.exists(file_path):
                    fastparquet.write(file_path, df, append=True)
                else:
                    fastparquet.write(file_path, df)

                n_success += 1
                print(f'{shot}: features extraction successful ({n_success} total)')

            except Exception as e:
                print(f'{shot}: feature extraction failed: {e}')
        
        return file_path


The model will use data from the ten basic parameters shown previously, using rollowing windows: the mean, gradient, minimum gradient, range and standard deviation will be the features used.

The labelling is using the condition that plasma current drops by 40% in the next 50ms, therefore what the model is predicting is whether this will be true.

In [ ]:
def evaluate_ml_model(feature_data_path: str, model: RandomForestClassifier, 
                      X_test: np.ndarray, y_test: np.ndarray, feature_cols: list,
                      zarr_path: str, test_shots: list):
    """Evaluates a trained Random Forest classifier using test data.
    Prints the ROC-AUC score and classification report, plots
    the ROC curve and top 15 feature importances, and plots probability
    alongside a random sample from test data.
    
    Parameters:
        model : RandomForestClassifier
            Trained Random Forest classifier.
        X_test : np.ndarray
            Feature matrix for the test shots.
        y_test : np.ndarray
            True labels for the test shots.
        feature_cols : list
            List of feature names corresponding to columns in X_test.
    
    Returns:
        None : None
    """

    # plots ROC
    y_proba = model.predict_proba(X_test)[:, 1]
    
    print(f"ROC-AUC: {roc_auc_score(y_test, y_proba):.3f}")
    print(classification_report(y_test, model.predict(X_test)))

    RocCurveDisplay.from_predictions(y_test, y_proba)
    plt.title('ROC Curve')
    plt.show()

    # plots important features
    importances = pd.Series(model.feature_importances_, index=feature_cols)
    importances.nlargest(15).plot(kind='barh')

    plt.title('Top 15 Feature Importances')


    random_shot = np.random.choice(test_shots)

    df = pd.read_parquet(feature_data_path)
    shot_id = random_shot

    shot_mask = df['shot_id'] == shot_id  # use the unbalanced full df
    X_shot = df.loc[shot_mask, feature_cols].values
    probs = model.predict_proba(X_shot)[:, 1]

    ds = xr.open_zarr(zarr_path, group=shot_id)
    ip = ds['ip']
    # get real time axis matching the features
    mask = ip > 0.7 * ip.median()
    start_time = ip.time.where(mask, drop=True).min()
    ip_trim = ip.sel(time=slice(start_time, None))

    # get real times from the zarr data that match the parquet rows
    real_times = ip_trim.time.values
    parquet_times = real_times[:len(probs)]  # trim to match probs length


    fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
    print(f"Selected shot: {random_shot}")

    # probability against time
    axes[0].plot(parquet_times, probs)
    axes[0].axhline(0.5, color='r', linestyle='--', label='threshold 0.5')
    axes[0].axhline(0.2, color='orange', linestyle='--', label='threshold 0.2')
    axes[0].set_ylabel('Plasma Drop Probability')
    axes[0].set_xlim(float(ip_trim.time.min()), float(ip_trim.time.max()))  # match x range
    axes[0].legend()

    #  ip against time
    axes[1].plot(ip_trim.time.values, ip_trim.values)
    axes[1].set_xlabel('Time (s)')
    axes[1].set_ylabel('Plasma Current (A)')

    fig.suptitle(f'{shot_id}')
    plt.tight_layout()
    plt.show()
    plotting_profiles(shot_id, zarr_path=zarr_path)

In [ ]:
def train_ml_model_RF(feature_data_path: str) -> None:
    """Runs the machine learning model, using a random
    forest classifier.
    
    Parameters:
        feature_data_path : str
            File path to where feature data is stored.
    
    Returns:
        model: RandomForestClassifier
            ML model trained on the data 
        X_test: np.ndarray
            The test shots the ML model hasn't seen
        y_test: np.ndarray
            Test labels the ML hasn't seen
        feature_cols: list
            list of the features the ML model used
    """

    df = pd.read_parquet(feature_data_path)

    # balance the dataset
    positive = df[df['label'] == True].sample(n=10000, random_state=41)
    negative = df[df['label'] == False].sample(n=20000, random_state=42)
    df = pd.concat([positive, negative]).sample(frac=1, random_state=42)  # shuffle

    print(f'Total rows: {len(df)}')
    print(f'Shots: {df['shot_id'].nunique()}')
    print(f'Label balance:\n{df['label'].value_counts()}')

    # split by shot not by row
    feature_cols = [c for c in df.columns if c not in ['label', 'shot_id', 'time']]
    shots = df['shot_id'].unique()
    shots = np.array(shots)  # convert from pyarrow to numpy
    train_shots, test_shots = train_test_split(shots, test_size=0.2, random_state=42)

    train_mask = df['shot_id'].isin(train_shots)
    test_mask = df['shot_id'].isin(test_shots)

    X_train = df.loc[train_mask, feature_cols].values
    y_train = df.loc[train_mask, 'label'].astype(int).values
    X_test = df.loc[test_mask, feature_cols].values
    y_test = df.loc[test_mask, 'label'].astype(int).values

    model = RandomForestClassifier(
    n_estimators=100,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1)

    model.fit(X_train, y_train)
    
    return model, X_test, y_test, feature_cols, test_shots


In [ ]:
def main_RF(shot_min: int, shot_max: int) -> None:
    """Gathers and organizes machine learning data, calls function to train
     model, and calls function to evaluate model.
    
    Parameters:
        shot_min : int
            Minimum shot ID.
        shot_max : int
            Maximum shot ID.
    
    Returns:
        None : None
    """

    # gets the zarr file path containing the unchecked data from shots
    zarr_path = read_file(shot_min, shot_max)
    print('Zarr path found')

    # gets the good data file path
    good_data_path = is_bad_data(zarr_path=zarr_path)
    print('Good data path found')

    # gets the file path for the file containing all the data information needed for the ml model
    feature_data_path = extracting_features(zarr_path=zarr_path, label_path=good_data_path)
    print('Features data path found')

    # trains model
    model, X_test, y_test, feature_cols, test_shots = train_ml_model_RF(feature_data_path)
    print('Model tested')

    # evaluates model
    evaluate_ml_model(feature_data_path, model, X_test, y_test, feature_cols, zarr_path, test_shots)
    print('Model evaluated')

In [ ]:
shot_min = 18500
shot_max = 30500

main_RF(shot_min, shot_max)

The ROC shows we currently have a 92% chance of correctly assigning shots with big plasma drops with higher probabilities. We also see the features importance, which shows WMHD and Li have the highest importances, however these can't be trusted entirely due to multicollinearity, furthermore, may suggest some label leakage.

In [ ]:
def train_ml_model_LGBM(feature_data_path: str):
    
    df = pd.read_parquet(feature_data_path)

    feature_cols = [c for c in df.columns if c not in ['label', 'shot_id', 'time']]
    shots = np.array(df['shot_id'].unique())
    train_shots, test_shots = train_test_split(shots, test_size=0.2, random_state=42)

    train_mask = df['shot_id'].isin(train_shots)
    test_mask = df['shot_id'].isin(test_shots)

    X_train = df.loc[train_mask, feature_cols].values
    y_train = df.loc[train_mask, 'label'].astype(int).values
    X_test = df.loc[test_mask, feature_cols].values
    y_test = df.loc[test_mask, 'label'].astype(int).values

    scale = len(df[df['label'] == False]) / len(df[df['label'] == True])

    model = LGBMClassifier(
        n_estimators=100,
        scale_pos_weight=scale,  # handles imbalance instead of resampling
        random_state=42,
        n_jobs=-1
    )

    model.fit(X_train, y_train)

    return model, X_test, y_test, feature_cols, test_shots

In [ ]:
def main_LGBM(shot_min: int, shot_max: int) -> None:
    """Gathers and organizes machine learning data, calls function to train
     model, and calls function to evaluate model.
    
    Parameters:
        shot_min : int
            Minimum shot ID.
        shot_max : int
            Maximum shot ID.
    
    Returns:
        None : None
    """

    # gets the zarr file path containing the unchecked data from shots
    zarr_path = read_file(shot_min, shot_max)
    print('Zarr path found')

    # gets the good data file path
    good_data_path = is_bad_data(zarr_path=zarr_path)
    print('Good data path found')

    # gets the file path for the file containing all the data information needed for the ml model
    feature_data_path = extracting_features(zarr_path=zarr_path, label_path=good_data_path)
    print('Features data path found')

    # trains model
    model, X_test, y_test, feature_cols, test_shots = train_ml_model_LGBM(feature_data_path)
    print('Model tested')

    # evaluates model
    evaluate_ml_model(feature_data_path, model, X_test, y_test, feature_cols, zarr_path, test_shots)
    print('Model evaluated')


In [ ]:
shot_min = 18500
shot_max = 30500

main_LGBM(shot_min, shot_max)

This model uses LightGBM, where trees learn from the past tree sequentially. This shows a slightly higher ROC, furthermore, has completely different feature importances. This cannot fairly be compared against RF however, as the entire dataset was used to train this model, rather than a sample like with RF. This was done because RF is much slower than LGBM. 

The results above show that the model is predicting about 50ms ahead, as can be seen by the probability peak relative to the time plasma current drops, showing the model is working effectively.